# BiLSTM-CRF Named Entity Recognition (NER) with POS Tagging
**IndoGist Machine Learning Pipeline — Experiment: `bilstm-crf with pos`**

This notebook documents and executes the end-to-end lifecycle for training a dual-input Part-of-Speech (POS) enhanced Bidirectional LSTM with Conditional Random Field (BiLSTM-CRF) model for Indonesian Named Entity Recognition (NER).

### Architecture & Pipeline Overview:
1. **Data Parsing**: Extracting words, POS tags, and NER labels from CoNLL-formatted datasets (`train.txt`, `dev.txt`, `test.txt`).
2. **Dual-Input Representations**:
   - **Word Embeddings**: 150-dimensional pre-trained Word2Vec embeddings fine-tuned during training.
   - **POS Embeddings**: 16-dimensional learned Part-of-Speech tag embeddings.
3. **BiLSTM-CRF Model**:
   - Word + POS Embeddings (166d) $\rightarrow$ 2-layer Bidirectional LSTM (64 units each, Dropout=0.2) $\rightarrow$ Dense Projection $\rightarrow$ Custom Keras CRF Layer with Viterbi decoding.
4. **Evaluation**: Token accuracy, Span-level Precision, Recall, and F1-Score via `seqeval` metrics.
5. **Artifact Export**: Exporting model weights (`best_model_by_f1.keras`) and metadata for Django `NLPService` integration.

## 1. Environment & GPU Setup (`tf-gpu` Docker Compatible)

In [ ]:
import os
import sys
import json
import pickle
import joblib
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import tensorflow as tf
import keras
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# Ensure repo root is on path
BASE_DIR = Path(".").resolve().parent
sys.path.insert(0, str(BASE_DIR))

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs Available: {gpus}")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU Memory Growth Enabled.")

## 2. Custom Keras CRF Layer & Loss Function

In [ ]:
from ml.ner.crf import CRFLayer, crf_loss, crf_accuracy, viterbi_decode_sample

print("Loaded custom CRF components cleanly:")
print(" - CRFLayer")
print(" - crf_loss")
print(" - crf_accuracy")
print(" - viterbi_decode_sample")

## 3. Data Loading & Vocabulary Building

In [ ]:
def load_conll_data(file_path):
    sentences = []
    current_words = []
    current_pos = []
    current_tags = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_words:
                    sentences.append((current_words, current_pos, current_tags))
                    current_words, current_pos, current_tags = [], [], []
                continue
            parts = line.split()
            if len(parts) >= 3:
                current_words.append(parts[0])
                current_pos.append(parts[1])
                current_tags.append(parts[2])
        if current_words:
            sentences.append((current_words, current_pos, current_tags))
            
    return sentences

train_path = BASE_DIR / "data" / "ner" / "train.txt"
dev_path = BASE_DIR / "data" / "ner" / "dev.txt"
test_path = BASE_DIR / "data" / "ner" / "test.txt"

train_data = load_conll_data(train_path)
dev_data = load_conll_data(dev_path)
test_data = load_conll_data(test_path)

print(f"Loaded {len(train_data)} train, {len(dev_data)} dev, {len(test_data)} test sentences.")

## 4. BiLSTM-CRF Model Construction (`bilstm-crf with pos`)

In [ ]:
from ml.ner.model import build_bilstm_crf_model

# Load mappings
with open(BASE_DIR / "data" / "ner" / "ner_pos_meta.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

vocab_size = meta.get("vocab_size", 10000)
pos_vocab_size = meta.get("pos_vocab_size", 30)
num_classes = meta.get("num_classes", 8)
max_len = meta.get("max_len", 50)

model = build_bilstm_crf_model(
    vocab_size=vocab_size,
    pos_vocab_size=pos_vocab_size,
    num_classes=num_classes,
    max_len=max_len,
    embedding_dim=150,
    pos_embedding_dim=16,
    lstm_units=64,
    dropout_rate=0.2
)

model.summary()

## 5. Model Evaluation on Test Dataset

In [ ]:
from ml.ner.evaluate import evaluate_bilstm_crf

model_dir = BASE_DIR / "ml" / "models" / "ner_pos_final"
test_file = BASE_DIR / "data" / "ner" / "test.txt"

metrics = evaluate_bilstm_crf(model_dir=str(model_dir), data_path=str(test_file))
print("\n--- Evaluation Metrics Summary ---")
print(json.dumps(metrics, indent=2))

## 6. End-to-End Service Prediction Test (`NLPService`)

In [ ]:
from ml.ner.predict import extract_entities

sample_text = "Joko Widodo mengunjungi Jakarta dan Bandung bersama Tentara Nasional Indonesia."
results = extract_entities(sample_text)
print("Sample Text:", sample_text)
print("Extracted Entities:", json.dumps(results, indent=2))